In [0]:
%sql
use catalog trueanalytics_data;

In [0]:
import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T 
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

def save_to_csv(df, save_path):
    (df.coalesce(1)
        .write.format('csv')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save csv to ", save_path)

In [0]:
# parameter: par_month
dbutils.widgets.text("par_month", "202515")
par_month = dbutils.widgets.get("par_month")

try:
  par_month = int(par_month)
except ValueError:
  par_month = 0
  raise ValueError("par_month value must be numeric")

if par_month!=0:
  pass
else:
  dbutils.notebook.exit("Aborting as ondition not met. Further tasks will be skipped")

# customer360 date
par_month_obj = datetime.strptime(str(par_month), '%Y%m')
next_par_month_obj = par_month_obj + relativedelta(months=1)
cust360_date = int(next_par_month_obj.strftime('%Y%m') + '01')

# debug
display('par_month = ',par_month)
display(' customer360_date = ',cust360_date)

In [0]:
# master data
prep_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_footfall.parquet'
prep_freq_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_flag_freq.parquet'
prep_feature_360 = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_360_feature.parquet'
profile_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/profile_chula.csv'
date_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/day_type_apr_june_26.csv'
nantional_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/country_group_chula.csv'
home_region_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/home_region_chula.csv'
h3_sub = 'gs://tdg-ds-tech-delivery/2026/chula/master/master_h3_sub_district.csv'
# report path
# report_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/report/'

report_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/report/report5/mask/{par_month}/'

In [0]:
df_date = (spark.read
      .option("header", "true").option("inferSchema", "true")
      .csv(date_path)).select('date','WEEK','day_type_final')

In [0]:
province_bmr = ['bangkok','nakhonpathom','nonthaburi','samutprakan','samutsakhon','pathumthani']
filter_bmr_home_province = (F.col('is_bmr')==1) & ((F.col("work_province").isin(province_bmr)) & (~F.col("home_province").isin(province_bmr)))
filter_bmr_work_province = (F.col('is_bmr')==1) & ((F.col("home_province").isin(province_bmr)) & (~F.col("work_province").isin(province_bmr)))

# par_day = ['20251101','20251102','20251103']
df = spark.read.parquet(prep_path)

df_freq = spark.read.parquet(prep_freq_path) # raw freq by mall
df_360 = spark.read.parquet(prep_feature_360)
     # raw profile
df_merge = df.join(F.broadcast(df_freq), ['msisdn','name'], 'left')\
    .join(F.broadcast(df_360.drop('is_bmr','is_non_bmr','is_foriegner','a_country_name','demo_tourist_sim_v1_tourist_bin','roaming_flag')), ['msisdn'], 'inner')
df_merge = df_merge\
    .join(df_date, [df_merge.par_day == df_date.date], "left")\
    .withColumnRenamed('par_month','month')

In [0]:
columns = ['msisdn'
           , F.col('geog_resident_location_v1_district_en_cat').alias('home_district')
           , F.col('geog_resident_location_v1_sub_district_en_cat').alias('home_subdistrict')]

df_cust360 = spark.read.table('trueanalytics_data.customer360.customer360_snapshot')\
    .filter((F.col('par_day')==cust360_date) & (F.col('activated_flag')=='1'))\
    .select(columns)


df_merge = df_merge.drop('home_district','home_subdistrict')\
    .join(F.broadcast(df_cust360), "msisdn", 'left')


df_merge = df_merge.withColumn("home_subdistrict", 
    F.when((F.col("home_province")=='bangkok') & (F.col("home_subdistrict")=='chantharakasem'), F.lit('chan kasem'))
    .otherwise(F.col('home_subdistrict'))
)

In [0]:
df_merge = df_merge.withColumnRenamed('par_month','month')\
    .withColumn("home_province",
                               F.when(filter_bmr_home_province,
                                      F.lit("non_bmr")).otherwise(F.col("home_province")))\
                    .withColumn("home_district",
                                F.when(filter_bmr_home_province,
                                       F.lit("non_bmr")).otherwise(F.col("home_district")))\
                    .withColumn("home_sub_district",
                                F.when(filter_bmr_home_province,
                                       F.lit("non_bmr")).otherwise(F.col("home_subdistrict")))\
                    .withColumn("work_province",
                               F.when(filter_bmr_work_province,
                                      F.lit("non_bmr")).otherwise(F.col("work_province")))\
                    .withColumn("work_district",
                                F.when(filter_bmr_work_province,
                                       F.lit("non_bmr")).otherwise(F.col("work_district")))\
                    .withColumn("work_sub_district",
                                F.when(filter_bmr_work_province,
                                       F.lit("non_bmr")).otherwise(F.col("work_subdistrict")))
display(df_merge.count())

In [0]:
print('total :',df_merge.select('msisdn').distinct().count())
print('bmr :',df_merge.filter(F.col('is_bmr')==1).select('msisdn').distinct().count())
print('non bmr :',df_merge.filter(F.col('is_non_bmr')==1).select('msisdn').distinct().count())
print('foriegner :',df_merge.filter(F.col('is_foriegner')==1).select('msisdn').distinct().count())
print('unknown :',df_merge.filter(((F.col('is_bmr')==0)&(F.col('is_non_bmr')==0)&(F.col('is_foriegner')==0))).select('msisdn').distinct().count()) # join 360 inner กรองออก

In [0]:
columns_to_update = [
    "home_province", "home_district", "home_subdistrict",
    "work_province", "work_district", "work_subdistrict"
]

for col in columns_to_update:
    df_merge = df_merge.withColumn(
        col,
        F.when(F.col(col).isNull(), F.lit("unidentified")).otherwise(F.col(col))
    )

In [0]:
df_merge = df_merge\
    .withColumnRenamed('region','home_region')\
    .withColumnRenamed('nationality_group','region')

In [0]:
select_col_mall = ['msisdn','gender','age_range','home_province','home_district','home_subdistrict','work_province','work_district','work_subdistrict','is_foriegner','is_bmr','is_non_bmr','day_type_final','monthly_pay','nationality','region','home_region',
'cpn_cbd_customers','cpn_cbd_frequent_customers','cpn_non_cbd_customers','cpn_non_cbd_frequent_customers','spw_iconics_customers','spw_iconics_frequent_customers','spw_spdscsd_customers','spw_spdscsd_frequent_customers','tcc_customers','tcc_frequent_customers','the_mall_cbd_customers','the_mall_cbd_frequent_customers','the_mall_non_cbd_customers','the_mall_non_cbd_frequent_customers','spw_group_customers','spw_group_frequent_customers','cpn_cbd_weekly_active','cpn_non_cbd_weekly_active','spw_iconics_weekly_active','spw_spdscsd_weekly_active','tcc_weekly_active','the_mall_cbd_weekly_active','the_mall_non_cbd_weekly_active','spw_group_weekly_active']

In [0]:
province_bmr = ['Bangkok','Nakhon Pathom','Nonthaburi','Samut Prakan','Samut Sakhon','Pathum Thani']
province_work_home_bmr = ['BANGKOK','SAMUTSAKHON','SAMUTPRAKAN','NAKHONPATHOM','NONTHABURI','PATHUMTHANI']

h3_loc_mapping = spark.read.table('trueanalytics_data.gdb_intel.h3_loc_mapping')\
    .filter(F.col('province').isin(province_bmr))

h3_sub_df = spark.read.csv(h3_sub, header=True)\
    .filter(F.col('error_case')!='h3_only')\
        .withColumn(
    "sub_district_360",
    F.when((F.col("district_360") == "vadhana") & (F.col("sub_district_h3") == "khlong tan nuea"), F.lit("khlong tan nuea"))
     .when((F.col("district_360") == "vadhana") & (F.col("sub_district_h3") == "khlong toei nuea"), F.lit("khlong toei nuea"))
     .when((F.col("district_360") == "vadhana") & (F.col("sub_district_h3") == "phra khanong nuea"), F.lit("phra khanong nuea"))
     .otherwise(F.col("sub_district_360"))
)

h3_loc_mapping_rename = h3_loc_mapping\
    .withColumn("province_h3", F.lower(F.col('province')))\
        .withColumn("district_h3", F.lower(F.col('district')))\
        .withColumn("sub_district_h3", F.lower(F.col('sub_district')))\
    .join(h3_sub_df, ['province_h3','district_h3','sub_district_h3'],'left')
    
h3_loc_mapping_rename = h3_loc_mapping_rename\
    .withColumn("province", F.when(F.col("error_case").isNotNull(), F.col("province_360")).otherwise(F.col("province_h3")))\
    .withColumn("district", F.when(F.col("error_case").isNotNull(), F.col("district_360")).otherwise(F.col("district_h3")))\
    .withColumn("sub_district", F.when(F.col("error_case").isNotNull(), F.col("sub_district_360")).otherwise(F.col("sub_district_h3")))
    
h3_geo_loc = spark.read.table('trueanalytics_data.geo_insights.h3_geo_loc')\
    .filter(F.col('par_month') == par_month)\
    .select('msisdn'
            ,'par_hour'
            ,'h3_index_res9'
            ,'par_day')\
    .join(F.broadcast(h3_loc_mapping_rename), 'h3_index_res9', 'inner')\
    .select('centroid_lat','centroid_lng','msisdn','par_hour','province','district','sub_district','par_day')
print(h3_geo_loc.select('par_day').distinct().count())
footfall = df_merge.select(select_col_mall).distinct()\
    .join(F.broadcast(h3_geo_loc),['msisdn'],'inner')

In [0]:
footfall_rename = footfall\
    .withColumnsRenamed({'centroid_lat':'latitude',
                                               'centroid_lng':'longitude',
                                               'day_type_final':'day_type'
                                               })\
                        .withColumn("time",
                            F.when(F.col("par_hour").between(0, 1), "00:01 - 02:00")
                             .when(F.col("par_hour").between(2, 5), "02:01 - 06:00")
                             .when(F.col("par_hour").between(6, 9), "06:01 - 10:00")
                             .when(F.col("par_hour").between(10, 13), "10:01 - 14:00")
                             .when(F.col("par_hour").between(14, 17), "14:01 - 18:00")
                             .when(F.col("par_hour").between(18, 21), "18:01 - 22:00")
                             .when(F.col("par_hour").between(22, 23), "22:01 - 00:00")
                             .otherwise("unidentified"))
                        
# footfall_rename.display()

In [0]:
# conditions
df_intermediate_placetype = footfall_rename\
        .withColumn('province_new', F.regexp_replace(F.col('province'), ' ', ''))\
        .withColumn('place_type', 
                        F.when(((F.col('home_province')==F.col('province_new'))
                                & (F.col('home_district')==F.col('district')) 
                                & (F.col('home_subdistrict')==F.col('sub_district'))) &
                                ((F.col('work_province')!=F.col('province_new')) 
                                 | (F.col('work_district')!=F.col('district')) 
                                 | (F.col('work_subdistrict')!=F.col('sub_district')) 
                                 | (F.col('work_subdistrict').isNull())), F.lit('home'))
                        .when(((F.col('work_province')==F.col('province_new'))
                                & (F.col('work_district')==F.col('district'))
                               & (F.col('work_subdistrict')==F.col('sub_district'))) &
                                ((F.col('home_province')!=F.col('province_new')) 
                                 | (F.col('home_district')!=F.col('district'))
                                 | (F.col('home_subdistrict')!=F.col('sub_district')) 
                                 | (F.col('home_subdistrict').isNull())), F.lit('work'))
                        
                        .when(((F.col('home_province')==F.col('province_new')) 
                                & (F.col('home_district')==F.col('district'))
                               & (F.col('home_subdistrict')==F.col('sub_district'))) &
                                ((F.col('work_province')==F.col('province_new'))
                                 & (F.col('work_district')==F.col('district')) 
                                 & (F.col('work_subdistrict')==F.col('sub_district'))), F.lit('home_work'))
                            
                        .when((((F.col('home_province')!=F.col('province_new')) 
                                | (F.col('home_district')!=F.col('district'))
                                | (F.col('home_subdistrict')!=F.col('sub_district'))) | 
                               (F.col('home_subdistrict').isNull())) &
                                (((F.col('work_province')!=F.col('province_new')) | 
                                  (F.col('work_district')!=F.col('district')) |
                                  (F.col('work_subdistrict')!=F.col('sub_district'))) | 
                                 (F.col('work_subdistrict').isNull())) &
                                ((F.col('home_subdistrict').isNotNull()) | 
                                 (F.col('work_subdistrict').isNotNull())), F.lit('thirdplace'))
                            
                        .when((F.col('home_province').isNull()) & (F.col('home_subdistrict').isNull()) &
                                (F.col('work_province').isNull()) & (F.col('work_subdistrict').isNull()), F.lit('home_work_unidentified'))
                        .otherwise(F.lit('other')) # debug
)

# debug: expect return 0 row
# display(df_intermediate_placetype.select('msisdn','place_type','work_province','work_subdistrict','home_province','home_subdistrict','province','sub_district').filter(F.col('place_type')=='other').limit(20))

In [0]:
r5t = ['latitude','longitude','province','district','sub_district','day_type','time']
df_bmr_r5t = df_intermediate_placetype.filter(F.col('is_bmr')==1)\
    .groupBy(r5t).agg(F.count('msisdn').alias('h3_frequency_visitor_cnt'))\
        .withColumn("h3_frequency_visitor_cnt", F.when(F.col("h3_frequency_visitor_cnt") <= 25, 25).otherwise(F.col("h3_frequency_visitor_cnt")))
df_non_bmr_r5t = df_intermediate_placetype.filter(F.col('is_non_bmr')==1)\
    .groupBy(r5t).agg(F.count('msisdn').alias('h3_frequency_visitor_cnt'))\
        .withColumn("h3_frequency_visitor_cnt", F.when(F.col("h3_frequency_visitor_cnt") <= 25, 25).otherwise(F.col("h3_frequency_visitor_cnt")))
df_foreigner_r5t = df_intermediate_placetype.filter(F.col('is_foriegner')==1)\
    .groupBy(r5t).agg(F.count('msisdn').alias('h3_frequency_visitor_cnt'))\
        .withColumn("h3_frequency_visitor_cnt", F.when(F.col("h3_frequency_visitor_cnt") <= 25, 25).otherwise(F.col("h3_frequency_visitor_cnt")))

save_to_csv(df_bmr_r5t, report_path+f"report5_bmr_r5t_{par_month}.csv")
save_to_csv(df_non_bmr_r5t, report_path+f"report5_non_bmr_r5t_{par_month}.csv")
save_to_csv(df_foreigner_r5t, report_path+f"report5_foreigner_r5t_{par_month}.csv")

In [0]:
r5p = ['latitude','longitude','province','district','sub_district','day_type','time','place_type']
df_bmr_r5p = df_intermediate_placetype.filter(F.col('is_bmr')==1)\
    .groupBy(r5p).agg(F.count('msisdn').alias('h3_frequency_visitor_cnt'))\
        .withColumn("h3_frequency_visitor_cnt", F.when(F.col("h3_frequency_visitor_cnt") <= 25, 25).otherwise(F.col("h3_frequency_visitor_cnt")))

save_to_csv(df_bmr_r5p, report_path+f"report5_bmr_r5p_{par_month}.csv")

In [0]:
bmr_r5f = ['latitude','longitude','province','district','sub_district','day_type','time','place_type','cpn_cbd_customers','cpn_non_cbd_customers','spw_iconics_customers','spw_spdscsd_customers','tcc_customers','the_mall_cbd_customers','the_mall_non_cbd_customers','spw_group_customers']
r5f = ['latitude','longitude','province','district','sub_district','day_type','time','cpn_cbd_customers','cpn_non_cbd_customers','spw_iconics_customers','spw_spdscsd_customers','tcc_customers','the_mall_cbd_customers','the_mall_non_cbd_customers','spw_group_customers']
df_bmr_r5f = df_intermediate_placetype.filter(F.col('is_bmr')==1)\
    .groupBy(bmr_r5f).agg(F.count('msisdn').alias('h3_frequency_visitor_cnt'))\
        .withColumn("h3_frequency_visitor_cnt", F.when(F.col("h3_frequency_visitor_cnt") <= 25, 25).otherwise(F.col("h3_frequency_visitor_cnt")))
df_non_bmr_r5f = df_intermediate_placetype.filter(F.col('is_non_bmr')==1)\
    .groupBy(r5f).agg(F.count('msisdn').alias('h3_frequency_visitor_cnt'))\
        .withColumn("h3_frequency_visitor_cnt", F.when(F.col("h3_frequency_visitor_cnt") <= 25, 25).otherwise(F.col("h3_frequency_visitor_cnt")))
df_foreigner_r5f = df_intermediate_placetype.filter(F.col('is_foriegner')==1)\
    .groupBy(r5f).agg(F.count('msisdn').alias('h3_frequency_visitor_cnt'))\
        .withColumn("h3_frequency_visitor_cnt", F.when(F.col("h3_frequency_visitor_cnt") <= 25, 25).otherwise(F.col("h3_frequency_visitor_cnt")))

save_to_csv(df_bmr_r5f, report_path+f"report5_bmr_r5f_{par_month}.csv")
save_to_csv(df_non_bmr_r5f, report_path+f"report5_non_bmr_r5f_{par_month}.csv")
save_to_csv(df_foreigner_r5f, report_path+f"report5_foreigner_r5f_{par_month}.csv")

In [0]:
core_columns = ['latitude','longitude','province','district','sub_district','day_type','time','gender','age_range','monthly_pay']
freq_columns = ['cpn_cbd_customers','cpn_cbd_frequent_customers','cpn_non_cbd_customers','cpn_non_cbd_frequent_customers','spw_iconics_customers','spw_iconics_frequent_customers','spw_spdscsd_customers','spw_spdscsd_frequent_customers','tcc_customers','tcc_frequent_customers','the_mall_cbd_customers','the_mall_cbd_frequent_customers','the_mall_non_cbd_customers','the_mall_non_cbd_frequent_customers','spw_group_customers','spw_group_frequent_customers','cpn_cbd_weekly_active','cpn_non_cbd_weekly_active','spw_iconics_weekly_active','spw_spdscsd_weekly_active','tcc_weekly_active','the_mall_cbd_weekly_active','the_mall_non_cbd_weekly_active','spw_group_weekly_active']
bmr = ['home_province','home_district','work_province','work_district','place_type']
non_bmr = ['home_region']
foreigner = ['nationality','region']

df_bmr = df_intermediate_placetype.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns+freq_columns+bmr).agg(F.count('msisdn').alias('h3_frequency_visitor_cnt'))\
        .withColumn("h3_frequency_visitor_cnt", F.when(F.col("h3_frequency_visitor_cnt") <= 25, 25).otherwise(F.col("h3_frequency_visitor_cnt")))
df_non_bmr = df_intermediate_placetype.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+freq_columns+non_bmr).agg(F.count('msisdn').alias('h3_frequency_visitor_cnt'))\
        .withColumn("h3_frequency_visitor_cnt", F.when(F.col("h3_frequency_visitor_cnt") <= 25, 25).otherwise(F.col("h3_frequency_visitor_cnt")))
df_foreigner = df_intermediate_placetype.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+freq_columns+foreigner).agg(F.count('msisdn').alias('h3_frequency_visitor_cnt'))\
        .withColumn("h3_frequency_visitor_cnt", F.when(F.col("h3_frequency_visitor_cnt") <= 25, 25).otherwise(F.col("h3_frequency_visitor_cnt")))

save_to_csv(df_bmr, report_path+f"report5_bmr_r5r_{par_month}.csv")
save_to_csv(df_non_bmr, report_path+f"report5_non_bmr_r5r_{par_month}.csv")
save_to_csv(df_foreigner, report_path+f"report5_foreigner_r5r_{par_month}.csv")